# Pycavi zero point calibration check

This notebook checks the calibration of swarped image calibrated in the previous step.

- Reads the swarped image and some necessary keywords.
- Read the catalogue used to calibrate the image.
- Produce several plots that can be used to determine the quality of the photometry.
- Saves the resulting plot into a file



In [ ]:
import os
import sys
import glob
import copy
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS

from photutils.aperture import SkyCircularAperture, SkyCircularAnnulus, aperture_photometry


#   from pycavi_astromatic import SExtractor
sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_tools
from pycavi.pycavi_zpcal import ZPCalibration, plot_calibration_check_frame


In [ ]:
# set the directory folders
config_dir = os.getcwd()+'/../config/'
config = pycavi_tools.read_pycavi_config(config_dir)

observatory = config['Provenance']['observatory'] # Needed for the atmospheric extinction curve
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
log_dir = config['Directories']['project_logs_dir']


# Set the target explicitly. At this point it is believed that
# the data reduction process should be closely monitored, so
# we calibrate target/filter sets one by one.
#
# Eventually the choice of target should be done via GUI
#
target = 'CP_Pup'
filter = 'r_SDSS'
reduced_dir = reduced_science_dir
sextractor_dir = reduced_dir+'/'+target+'/'+config['Directories']['sex_dir_name']
swarp_dir = reduced_dir+'/'+target+'/'+config['Directories']['swarp_dir_name']
calibration_dir = reduced_dir+'/'+target+'/'+config['Directories']['cal_dir_name']

log_file_swarp = log_dir + "/pymeffpp_swarp.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# We create a list with the names of the registered and combined
# images output by the notebook 6 swarp. 
#
#images_to_calibrate = []
#for image in glob.glob(swarp_dir + "/*.list"):
#    images_to_calibrate.append(image[:-5]+".coadd.fits")
print(swarp_dir)
images_to_check = glob.glob(swarp_dir + '/*_'+filter+'.coadd.fits')
images_to_check.sort()

for image in images_to_check:

    plot_calibration_check_frame(image, calibration_dir, swarp_dir)
    
logging.info("-------Done-------")